# Price Asymmetric Strategy (PAS)
QUANTIFI · 2026 Fall · Team research

기존 계산 코드 및 변수명은 유지하고 포트폴리오 열람을 위해 Markdown 설명만 추가했다. 실행 결과와 실행 메타데이터는 개인정보·원천 데이터 보호를 위해 제거했다. 입력 CSV는 DataGuide 이용 권한이 필요하며 저장소에 제공하지 않는다. 독립 재실행은 수행하지 않았다.

## 1. 데이터 로딩·전처리·팩터 구성 및 기본 백테스트

DataGuide의 수정주가·배당반영 가격·시총·ADV20·요인 데이터를 정렬하고, 일간·월간 시계열의 용도에 맞게 통계량을 계산한다. 월별 유니버스, 순위 기반 신호, 숏 우선 선정, 체결률 및 비용을 적용한다. 원 코드의 미래 가격 존재 여부 필터와 상폐 수익률 처리는 별도 검토 대상이다.

In [ ]:
# ============================================================================
# 기본적인 코드 흐름
# - 시가총액 상위 30% 안에서 long_score 상위 50개와 short_score 하위 100개를 롱,숏 후보로선정한다.
# - 후보로 선정한 종목 중 체결률 80% 조건을 통과한 종목만 운용한다. 
# - 동일가중, 롱 110%, 숏 90%로 운용하며 gross 는 항상 200%를 유지한다. 
# - 최종 수익률 계산 시 거래비용과 숏 대차비용을 모두 반영해 계산한다. 
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# [1] 입력 데이터 파일
# price: 배당을 제외한 수정주가로 모멘텀, 변동성, 왜도, Sharpe 계산에 사용한다.
# total: 현금배당을 포함한 수정주가로 수익률 계산에 사용한다.
# cap: 시가총액으로 유니버스 구성 시 사용한다.
# adv: 20거래일 평균 거래대금으로 유동성 제약 계산 시 사용한다.
# factor: KOSPI와 무위험수익률(Rf) 등 시장 팩터 자료다.
price_file = "adj_close_0906.csv"
total_file = "adj_close_dividend_0906.csv"
cap_file = "mkt_cap_0906.csv"
adv_file = "20일평균거래대금.csv"
factor_file = "Factor_0917.csv"

# [2] 기본 조건
# 성과 측정 기간은 2016년 9월부터 2026년 8월까지 총 120개월이다.
# 목표 종목 수는 롱 50개, 숏 100개이며 실제 보유 수는 유동성 검사 후 줄어들 수 있다.
# 포트폴리오 노출은 롱 110%, 숏 90%이므로 Gross 200%, Net +20%다.
months = pd.period_range("2016-09", "2026-08", freq="M")
long_n, short_n = 50, 100
long_weight, short_weight = 1.10, 0.90


# [3] 패널 데이터 로딩 함수
# 첫 번째 열을 날짜 인덱스로 읽고 모든 종목 값을 숫자로 변환한다.
# 숫자로 바꿀 수 없는 값은 NaN으로 처리하며 2026-08-31 이후 자료는 사용하지 않는다.
# start가 지정되면 장기 팩터 산출에 필요한 시작 시점 이후만 남긴다.
def read_panel(filename, start=None):
    data = pd.read_csv(filename, index_col=0, parse_dates=True)
    data = data.apply(pd.to_numeric, errors="coerce").loc[:"2026-08-31"]
    return data.loc[start:] if start else data


# 가격·거래대금은 백테스트 시작보다 앞선 2013년 1월부터 불러온다.
price = read_panel(price_file, "2013-01-01")
total = read_panel(total_file, "2013-01-01")
cap = read_panel(cap_file)
adv = read_panel(adv_file, "2013-01-01")

# 네 데이터셋에 모두 존재하는 종목만 남겨 종목 코드 불일치로 인한 오류를 방지한다.
names = price.columns.intersection(total.columns).intersection(cap.columns).intersection(adv.columns)
price, total, cap, adv = [
    data.reindex(columns=names) for data in (price, total, cap, adv)
]

# [4] 월말 자료 정렬
# 달력상 말일이 아니라 실제 가격 자료에 존재하는 각 월의 마지막 거래일을 사용한다.
ends = price.index.to_series().groupby(price.index.to_period("M")).last()
dates = pd.DatetimeIndex(ends.loc["2014-01":"2026-08"].values)
date_of = dict(zip(dates.to_period("M"), dates))

# 수정주가는 실제 월말 거래일 값을 사용한다.
# 시가총액·거래대금·배당포함 수정주가는 월말 값이 없을 경우 그 이전 최신값을 전방 채움한다.
# 형성일 이후 정보를 당겨 쓰지 않음으로서 look-ahead bias를 방지한다. 
monthly_price = price.reindex(dates)
monthly_total = total.reindex(total.index.union(dates)).sort_index().ffill().reindex(dates)
monthly_cap = cap.reindex(cap.index.union(dates)).sort_index().ffill().reindex(dates)
monthly_adv = adv.reindex(adv.index.union(dates)).sort_index().ffill().reindex(dates)

# [5] 팩터 계산용 일간 수익률
# 배당 미포함 수정주가를 사용해 배당락의 기계적 가격 변화를 팩터에서 분리한다.
# 결측치를 자동 전방 채움하지 않으며 0으로 나눈 무한대 값은 NaN으로 바꾼다.
daily_ret = price.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)

# MOM12 = P(t-1) / P(t-12) - 1. 점수에는 직접 쓰지 않지만,
# 아래 stocks.dropna()에 포함되어 최소 12개월 가격 이력이 있는 종목만 남기는 역할을 한다.
# MOM6 = P(t-1) / P(t-6) - 1. 최근 1개월을 제외한 중기 모멘텀으로 롱·숏 점수에 사용한다.
mom12 = monthly_price.shift(1) / monthly_price.shift(12) - 1
mom6 = monthly_price.shift(1) / monthly_price.shift(6) - 1

# 최근 126거래일(약 6개월)의 일간 수익률 통계량을 계산한다.
# 최소 100개 관측치가 있어야 변동성(VOL126), 왜도(SKEW126), 평균을 인정한다.
rolling = daily_ret.rolling(126, min_periods=100)
vol = rolling.std().reindex(dates)
skew = rolling.skew().reindex(dates)
mean = rolling.mean().reindex(dates)

# 롱 후보의 품질 필터: 최근 126거래일 중 음(-)의 수익률을 기록한 날이
# 최소 60일 이상인지 확인한다. 하락일이 너무 적은 저유동성·단기 급등 종목에서
# 변동성이 과소 측정되어 Sharpe가 비정상적으로 커지는 문제를 줄이기 위한 조건이다.
fall_ok = (
    daily_ret.where(daily_ret < 0)
    .rolling(126, min_periods=60)
    .std()
    .reindex(dates)
    .notna()
)


# 횡단면 백분위 순위 함수. 값이 작을수록 0에, 클수록 1에 가까우며
# 동일한 값은 평균 순위를 부여한다. 모든 점수는 절대값 대신 유니버스 내 상대순위를 쓴다.
def rank(series):
    return series.rank(pct=True, method="average")


# [6] 종목 선정 함수
# date(형성 월말)의 정보만으로 후보를 고르고 next_date 자료 존재 여부를 확인한다.
# 기본 체결 기간은 5거래일이며 강건성 검증에서는 3일과 10일도 사용한다.
def pick(date, next_date, execution_days=5):
    # 형성 시점 시가총액의 횡단면 순위가 70% 이상인 종목, 즉 상위 30%만 투자 대상으로 삼는다.
    market_cap = monthly_cap.loc[date].dropna()
    universe = market_cap[market_cap.rank(pct=True) >= 0.70].index

    # MOM12, MOM6, VOL126, SKEW126, 평균수익률이 모두 계산되는 종목만 남긴다.
    # MOM12는 점수 항목은 아니지만 충분한 상장·가격 이력을 요구하는 자료 요건이다.
    stocks = pd.DataFrame({
        "mom12": mom12.loc[date],
        "mom6": mom6.loc[date],
        "vol": vol.loc[date],
        "skew": skew.loc[date],
        "mean": mean.loc[date],
    }).dropna()

    # 팩터 자료 요건과 시가총액 상위 30% 조건을 동시에 만족하는 최종 유니버스를 구성한다.
    stocks = stocks.loc[stocks.index.intersection(universe)]
    # 유효 가격 조건을 순서대로 적용해 실제 수익률 계산이 가능한 종목만 유지한다.
    stocks = stocks.loc[
        price.loc[date].reindex(stocks.index).gt(0)
        & total.loc[date].reindex(stocks.index).gt(0)
    ]
    # 유효 가격 조건을 순서대로 적용해 실제 수익률 계산이 가능한 종목만 유지한다.
    stocks = stocks.loc[
        # 첫 조건은 형성일의 두 수정주가가 모두 양수인지 확인하고,
        # 이 조건은 다음 월말 총수익 가격이 존재하는지 확인한다.
        monthly_total.loc[next_date].reindex(stocks.index).notna()
    ]

    # SHARPE126은 최근 126거래일의 일평균 수익률을 일간 변동성으로 나눈 값이다.
    # 변동성이 0이면 정의할 수 없으므로 NaN으로 처리한다.
    stocks["sharpe"] = stocks["mean"] / stocks["vol"].replace(0, np.nan)
    # 롱 점수 = rank(MOM6) × rank(SHARPE126) × [1 - rank(SKEW126)].
    # 모멘텀과 위험조정 성과가 높고, 양(+)의 왜도가 낮아 소수 급등일에 의존하지 않는
    # "고르게 오른 종목"일수록 점수가 높아진다.
    # 설명서의 기하평균 표현에서 세제곱근은 단조증가 함수라 순위를 바꾸지 않으므로 생략했다.
    stocks["long_score"] = (
        rank(stocks["mom6"])
        * rank(stocks["sharpe"])
        * (1 - rank(stocks["skew"]))
    ) 

    # 숏 점수 = rank(MOM6) × [1 - rank(VOL126)] × [1 - rank(SKEW126)].
    # 모멘텀이 낮거나 변동성·양의 왜도가 큰 "복권형 종목"일수록 곱이 작아진다.
    # 따라서 숏은 이 점수의 하위 종목을 선택한다.
    stocks["short_score"] = (
        rank(stocks["mom6"])
        * (1 - rank(stocks["vol"]))
        * (1 - rank(stocks["skew"]))
    ) 

    # 선정 순서 1: 숏 점수가 가장 낮은 100개를 먼저 숏 후보로 지정한다.
    # 숏 후보에 우선권을 주어 복권형 종목이 동시에 롱에 포함되지 않도록 한다.
    shorts_before = list(stocks["short_score"].nsmallest(short_n).index)
    # 선정 순서 2: SHARPE126이 유효하고 하락일 60일 조건을 만족하는 종목만 롱 후보 자격을 얻는다.
    long_ok = (
        stocks["sharpe"].notna()
        & fall_ok.loc[date].reindex(stocks.index).fillna(False)
    )
    # 선정 순서 3: 숏 후보를 제외한 뒤 롱 점수 상위 50개를 롱 후보로 지정한다.
    longs_before = list(
        stocks.loc[long_ok, "long_score"]
        .drop(shorts_before, errors="ignore")
        .nlargest(long_n)
        .index
    )

    # [7] 유동성 및 체결 가능성 검사
    # 레그별 운용금액 1,000억원을 목표 종목 수로 나누면 종목당 주문금액은
    # 롱 20억원(1,000억/50), 숏 10억원(1,000억/100)이다.
    # 하루에 20일 평균 거래대금의 10%까지만 시장 충격 없이 체결한다고 가정한다.
    liquidity = monthly_adv.loc[date].reindex(stocks.index).fillna(0).clip(lower=0)
    # 체결률 = min(1, ADV20 × 10% × 체결일수 / 종목당 목표 주문금액).
    # 기준 5일일 때 요구되는 ADV20 하한은 롱 32억원, 숏 16억원이다.
    short_fill = (liquidity * 0.10 * execution_days / (100_000_000_000 / short_n)).clip(upper=1)
    long_fill = (liquidity * 0.10 * execution_days / (100_000_000_000 / long_n)).clip(upper=1)

    # 후보 중 예상 체결률이 80% 이상인 종목만 최종 편입한다.
    # 탈락한 자리는 다음 순위 종목으로 채우지 않으므로 실제 종목 수는 50개·100개보다 적을 수 있다.
    longs = [name for name in longs_before if long_fill[name] >= 0.80]
    shorts = [name for name in shorts_before if short_fill[name] >= 0.80]

    # 유동성 검사를 통과한 최종 롱·숏 종목 목록을 반환한다.
    return longs, shorts


# [8] 분기 리밸런싱과 월별 수익률 계산
# 종목은 3개월마다 바꾸되 수익률은 매월 계산한다.
# 첫 형성일은 2016-08 월말이며 이후 매년 2·5·8·11월 말에 종목을 다시 선정하는 구조다.
returns = []
old_long = None
old_short = None

# 총 120개월을 순회한다. 분기 중간 달에는 직전 리밸런싱에서 선정한 종목을 그대로 사용한다.
for i, month in enumerate(months):
    # 해당 월 수익률은 직전 월 마지막 거래일(start)부터 현재 월 마지막 거래일(end)까지 계산한다.
    start = date_of[month - 1]
    end = date_of[month]
    fee = 0.0

    # 0, 3, 6, ... 번째 달에만 종목을 다시 선정하므로 리밸런싱 주기는 분기다.
    if i % 3 == 0:
        longs, shorts = pick(start, end)

        # 유동성 검사 후 남은 종목끼리 각 레그 내부에서 동일가중한다.
        # 따라서 종목이 탈락해도 현금을 보유하지 않고 남은 종목의 비중이 커진다.
        w_long = pd.Series(1 / len(longs), index=longs)
        w_short = pd.Series(1 / len(shorts), index=shorts)

        # 회전율은 신·구 동일가중 벡터 차이의 절댓값 합(Σ|Δw|)이다.
        # 첫 포트폴리오 구성은 기존 보유가 없으므로 각 레그 회전율을 1로 둔다.
        long_turn = (
            1.0 if old_long is None
            else w_long.sub(old_long, fill_value=0).abs().sum()
        )
        short_turn = (
            1.0 if old_short is None
            else w_short.sub(old_short, fill_value=0).abs().sum()
        )
        # 매매비용은 편도 25bp × (롱 노출×롱 회전율 + 숏 노출×숏 회전율)로 계산한다.
        # 종목 교체가 없는 분기 중간 달에는 fee가 0이다.
        fee = 0.0025 * (
            long_weight * long_turn + short_weight * short_turn
        )

        # 다음 리밸런싱의 회전율 계산을 위해 현재 목표 비중을 저장한다.
        old_long = w_long
        old_short = w_short

    # 보유기간 수익률은 현금배당을 포함한 수정주가로 계산한다.
    # 각 레그 수익률은 해당 월에 가격이 존재하는 편입 종목들의 단순평균이다.
    stock_returns = monthly_total.loc[end] / monthly_total.loc[start] - 1
    long_return = stock_returns.reindex(longs).dropna().mean()
    short_return = stock_returns.reindex(shorts).dropna().mean()

    # 월 순수익률 = 1.1×롱 수익률 - 0.9×숏 수익률 - 매매비용 - 월 대차비용.
    # 숏 대차비용은 연 3%를 단순 월할하여 숏 노출 90%에 적용한다(월 0.225%).
    net_return = (
        long_weight * long_return
        - short_weight * short_return
        - fee
        - 0.03 * short_weight / 12
    )
    returns.append((end, net_return))

# 날짜별 전략 순수익률을 하나의 월별 시계열로 변환한다.
strategy = pd.Series(dict(returns), name="롱숏 전략")

# [9] 비교 지수
# Factor_0917.csv에서 KOSPI 일별 지수를 읽어 같은 월말 기준 비교 수익률을 만든다.
factor = pd.read_csv(
    factor_file, index_col=0, parse_dates=True
).apply(pd.to_numeric, errors="coerce").sort_index()

# KOSPI의 각 월 마지막 관측치를 사용해 월 수익률을 계산한다.
kospi = factor["KOSPI"].resample("ME").last().pct_change()
kospi.index = kospi.index.to_period("M")

# 전략과 동일한 120개월 날짜 인덱스에 KOSPI 수익률을 맞춘다.
benchmark = pd.Series(
    kospi.reindex(strategy.index.to_period("M")).to_numpy(),
    index=strategy.index,
    name="KOSPI",
)


# [10] 성과지표 함수
# CAGR: 전체 누적수익을 실제 월수로 연환산한다.
# 연변동성: 월 수익률 표준편차에 √12를 곱한다.
# Sharpe: 이 첫 표에서는 무위험수익률을 차감하지 않은 단순 연환산 값이다.
# MDD: 누적 NAV가 과거 고점에서 가장 크게 하락한 비율이다.
def performance(returns):
    # 비교 가능한 유효 월만 사용하고 월 수익률을 복리 누적해 NAV를 만든다.
    returns = returns.dropna()
    nav = (1 + returns).cumprod()

    return pd.Series({
        "CAGR": nav.iloc[-1] ** (12 / len(returns)) - 1,
        "연변동성": returns.std() * np.sqrt(12),
        "Sharpe": returns.mean() / returns.std() * np.sqrt(12),
        "MDD": (nav / nav.cummax() - 1).min(),
    })


# 전략과 KOSPI의 성과지표를 열 방향으로 합친다.
# 이 result는 아래 강건성 표(result_rf)로 표시를 대체하므로 별도로 display하지 않는다.
result = pd.concat(
    [
        performance(strategy).rename("롱숏 전략"),
        performance(benchmark).rename("KOSPI"),
    ],
    axis=1,
)

# 월 수익률을 누적해 시작값 1의 NAV로 변환하고 전략과 KOSPI를 비교한다.
nav = pd.concat(
    [(1 + strategy).cumprod(), (1 + benchmark).cumprod()],
    axis=1,
)
nav.plot(figsize=(11, 5), linewidth=2, title="NAV: 롱숏 전략 vs KOSPI")
plt.ylabel("NAV (시작값 1)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 체결 기간 민감도 분석

동일한 전략을 3일·5일·10일 집행 가정으로 비교한다. 체결 가능성 및 성과의 관계를 점검한다.

In [ ]:
# ============================================================================
# [11] 체결 기간 강건성 검증
# 동일한 전략을 3일, 5일, 10일 체결 가정으로 반복 실행한다.
# execution_days가 길수록 ADV20으로 체결 가능한 금액이 커져 더 많은 후보가 남는다.
# 종목 선정·동일가중·분기 리밸런싱·비용 구조는 위 기본 백테스트와 동일하다.
# ============================================================================
def backtest(execution_days):
    # 각 체결 기간별로 월 순수익률과 직전 리밸런싱 비중을 독립적으로 관리한다.
    returns = []
    old_long = None
    old_short = None
    longs, shorts = [], []
    for i, month in enumerate(months):
        start = date_of[month - 1]
        end = date_of[month]
        fee = 0.0
        # 분기 첫 달에만 해당 execution_days를 적용해 유동성 검사를 다시 수행한다.
        if i % 3 == 0:
            longs, shorts = pick(start, end, execution_days)
            # 체결률 80% 기준을 통과한 최종 종목끼리 레그 내부 동일가중한다.
            w_long = pd.Series(1 / len(longs), index=longs)
            w_short = pd.Series(1 / len(shorts), index=shorts)
            long_turn = 1.0 if old_long is None else w_long.sub(old_long, fill_value=0).abs().sum()
            short_turn = 1.0 if old_short is None else w_short.sub(old_short, fill_value=0).abs().sum()
            # 리밸런싱 월에만 편도 25bp의 회전율 기반 매매비용을 차감한다.
            fee = 0.0025 * (long_weight * long_turn + short_weight * short_turn)
            old_long, old_short = w_long, w_short
        # 배당 포함 수정주가로 종목별 월 총수익률을 계산한다.
        stock_returns = monthly_total.loc[end] / monthly_total.loc[start] - 1
        long_return = stock_returns.reindex(longs).dropna().mean()
        short_return = stock_returns.reindex(shorts).dropna().mean()
        # 롱 110% 수익에서 숏 90% 수익, 매매비용, 연 3%의 월할 대차비용을 차감한다.
        net_return = (long_weight * long_return - short_weight * short_return - fee - 0.03 * short_weight / 12)
        returns.append((end, net_return))
    return pd.Series(dict(returns), name=f"{execution_days}일 전략")

# 3일·5일·10일의 세 시나리오를 실행한다.
# 설명서 기준 결과는 체결 기간이 길어질수록 종목 수가 늘지만 Sharpe가 대체로 유지되는지 확인하는 것이다.
strategies = {days: backtest(days) for days in (3, 5, 10)}
# 성과 비교를 위해 KOSPI 월 수익률과 월별 무위험수익률을 같은 팩터 파일에서 다시 구성한다.
factor = pd.read_csv(factor_file, index_col=0, parse_dates=True).apply(pd.to_numeric, errors="coerce").sort_index()
kospi = factor["KOSPI"].resample("ME").last().pct_change()
kospi.index = kospi.index.to_period("M")
benchmark = pd.Series(kospi.reindex(strategies[5].index.to_period("M")).to_numpy(), index=strategies[5].index, name="KOSPI")

# Rf는 연율(%)이므로 먼저 소수 연수익률로 바꾸고 252거래일 복리 기준 일수익률로 환산한다.
# 이후 월 안의 일수익률을 복리 누적해 월 무위험수익률을 만든다.
daily_rf = (1 + factor["Rf"] / 100) ** (1 / 252) - 1
rf_monthly = (1 + daily_rf).resample("ME").prod() - 1
rf_monthly.index = rf_monthly.index.to_period("M")

# [12] 무위험수익률을 반영한 최종 성과지표
# Sharpe = 월 초과수익률(전략 수익률 - 월 Rf)의 평균 / 월 수익률 표준편차 × √12.
# 분모는 전략의 월 수익률 표준편차이며 CAGR·연변동성·MDD 정의는 위와 같다.
def performance_with_rf(returns):
    returns = returns.dropna()
    nav = (1 + returns).cumprod()
    # 각 전략 수익률 월과 동일한 월의 Rf를 위치 기준 배열로 맞춘다.
    rf = rf_monthly.reindex(returns.index.to_period("M")).to_numpy()
    excess = returns.to_numpy() - rf
    return pd.Series({
        "CAGR": nav.iloc[-1] ** (12 / len(returns)) - 1,
        "연변동성": returns.std() * np.sqrt(12),
        "Sharpe (Rf 반영)": excess.mean() / returns.std() * np.sqrt(12),
        "MDD": (nav / nav.cummax() - 1).min(),
    })

# 체결 기간별 전략과 KOSPI의 CAGR, 연변동성, Rf 반영 Sharpe, MDD를 한 표로 비교한다.
result_rf = pd.concat([performance_with_rf(strategies[days]).rename(f"{days}일 전략") for days in (3, 5, 10)] + [performance_with_rf(benchmark).rename("KOSPI")], axis=1)
# 소수점 넷째 자리까지 표시한다. 저장된 기준 5일 결과는 CAGR 0.2158, Sharpe 1.1370, MDD -0.1364다.
display(result_rf.round(4))

## 3. Rolling IC 및 Signal Decay

형성 시점 점수와 이후 수익률의 횡단면 순위 상관으로 신호 예측력을 평가한다. 미래 수익률은 사후 검증 대상이다.

In [ ]:
# ============================================================
# Rolling IC / Signal Decay 분석
# 목적:
# 1) Long Score / Short Score가 미래 수익률을 얼마나 예측하는지 확인
# 2) 신호가 1개월 후에만 유효한지, 2~3개월까지 유지되는지 확인
# 3) 월별 vs 분기 리밸런싱 주기 선택의 근거 마련
# ============================================================

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt




# ------------------------------------------------------------
# 1. 한 시점의 Long / Short Score 계산
# ------------------------------------------------------------

def get_scores(date):
    """
    해당 월말까지 이용 가능한 정보만 사용해서
    Long Score와 Short Score를 계산한다.

    IC 분석 목적이므로:
    - 유동성 필터 적용 전
    - 상위 50 / 하위 100 선정 전
    - next_date 정보 사용 안 함
    """

    # 시가총액 상위 30% 유니버스
    market_cap = monthly_cap.loc[date].dropna()

    universe = market_cap[
        market_cap.rank(pct=True) >= 0.70
    ].index

    # 팩터 데이터 구성
    stocks = pd.DataFrame({
        "mom6": mom6.loc[date],
        "vol": vol.loc[date],
        "skew": skew.loc[date],
        "mean": mean.loc[date],
    }).dropna()

    # 시총 상위 30%만
    stocks = stocks.loc[
        stocks.index.intersection(universe)
    ]

    # 형성 시점에 실제 가격 존재
    valid_now = (
        price.loc[date]
        .reindex(stocks.index)
        .gt(0)
    )

    stocks = stocks.loc[valid_now]

    # 126일 Sharpe
    stocks["sharpe"] = (
        stocks["mean"]
        / stocks["vol"].replace(0, np.nan)
    )

    # --------------------------------------------------------
    # Long Score
    # High MOM6 + High Sharpe + Low Skew
    # --------------------------------------------------------

    stocks["long_score"] = (
        rank(stocks["mom6"])
        * rank(stocks["sharpe"])
        * (1 - rank(stocks["skew"]))
    )

    # --------------------------------------------------------
    # Short Score
    # Low MOM6 + High VOL + High SKEW 일수록 점수가 낮음
    # --------------------------------------------------------

    stocks["short_score"] = (
        rank(stocks["mom6"])
        * (1 - rank(stocks["vol"]))
        * (1 - rank(stocks["skew"]))
    )

    # Long은 실제 전략과 동일하게
    # 하락일 60일 이상 조건 적용
    long_valid = (
        fall_ok.loc[date]
        .reindex(stocks.index)
        .fillna(False)
    )

    stocks["long_valid"] = long_valid

    return stocks


# ------------------------------------------------------------
# 2. Forward Return 함수
# ------------------------------------------------------------

def get_forward_return(date, months_ahead):
    """
    date 시점에서 months_ahead개월 뒤까지의 누적수익률

    예:
    1M = t -> t+1
    2M = t -> t+2
    3M = t -> t+3
    """

    period = date.to_period("M")
    future_period = period + months_ahead

    if future_period not in date_of:
        return None

    future_date = date_of[future_period]

    ret = (
        monthly_total.loc[future_date]
        / monthly_total.loc[date]
        - 1
    )

    return ret


# ------------------------------------------------------------
# 3. 월별 Cross-sectional IC 계산
# ------------------------------------------------------------

ic_records = []

# 최소 3개월 후 수익률이 있어야 하므로
# 마지막 3개월은 자동 제외
for date in dates:

    # 실제 백테스트 직전부터 분석
    if date < pd.Timestamp("2016-08-01"):
        continue

    scores = get_scores(date)

    for horizon in [1, 2, 3]:

        fwd_ret = get_forward_return(date, horizon)

        if fwd_ret is None:
            continue

        # ====================================================
        # LONG IC
        # Long Score가 높을수록 미래수익률이 높아야 함
        # -> IC > 0 이 좋은 결과
        # ====================================================

        long_df = pd.concat(
            [
                scores.loc[
                    scores["long_valid"],
                    "long_score"
                ],
                fwd_ret.rename("future_return")
            ],
            axis=1
        ).dropna()

        if len(long_df) >= 30:

            long_ic = long_df[
                "long_score"
            ].corr(
                long_df["future_return"],
                method="spearman"
            )

        else:
            long_ic = np.nan


        # ====================================================
        # SHORT IC
        #
        # Short Score는 낮을수록 숏 방향.
        #
        # 따라서:
        # 낮은 short_score -> 낮은 미래수익률
        # 높은 short_score -> 높은 미래수익률
        #
        # 즉 raw short_score와 미래수익률의 IC는
        # 양수(+)일수록 전략 논리와 일치
        # ====================================================

        short_df = pd.concat(
            [
                scores["short_score"],
                fwd_ret.rename("future_return")
            ],
            axis=1
        ).dropna()

        if len(short_df) >= 30:

            short_ic = short_df[
                "short_score"
            ].corr(
                short_df["future_return"],
                method="spearman"
            )

        else:
            short_ic = np.nan


        ic_records.append({
            "date": date,
            "horizon": horizon,
            "long_ic": long_ic,
            "short_ic": short_ic,
            "n_long": len(long_df),
            "n_short": len(short_df),
        })


ic_df = pd.DataFrame(ic_records)

ic_df["date"] = pd.to_datetime(
    ic_df["date"]
)

ic_df.head()

## 4. IC 요약표

예측 기간별 IC의 평균과 변동성을 집계한다.

In [ ]:
# ============================================================
# 4. IC Decay Table
# ============================================================

ic_summary = (
    ic_df
    .groupby("horizon")
    .agg(
        Long_IC=("long_ic", "mean"),
        Short_IC=("short_ic", "mean"),

        Long_IC_Median=("long_ic", "median"),
        Short_IC_Median=("short_ic", "median"),

        Long_IC_Positive_Rate=(
            "long_ic",
            lambda x: (x > 0).mean()
        ),

        Short_IC_Positive_Rate=(
            "short_ic",
            lambda x: (x > 0).mean()
        ),

        N=("date", "count"),
    )
)

ic_summary.index = [
    "1개월 후",
    "2개월 후",
    "3개월 후"
]

print("\n===== IC DECAY =====")
display(ic_summary.round(4))

## 5. IC Decay Curve

미래 수익률 측정 기간별 평균 IC를 시각화한다.

In [ ]:
# ============================================================
# 5. IC Decay Curve
# ============================================================

decay = (
    ic_df
    .groupby("horizon")[
        ["long_ic", "short_ic"]
    ]
    .mean()
)

plt.figure(figsize=(8, 5))

plt.plot(
    decay.index,
    decay["long_ic"],
    marker="o",
    linewidth=2,
    label="Long Score"
)

plt.plot(
    decay.index,
    decay["short_ic"],
    marker="o",
    linewidth=2,
    label="Short Score"
)

plt.axhline(
    0,
    linewidth=1
)

plt.xticks(
    [1, 2, 3],
    ["1M", "2M", "3M"]
)

plt.xlabel("Forward Return Horizon")
plt.ylabel("Average Spearman IC")

plt.title(
    "Signal Decay: 1M / 2M / 3M Forward IC"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 12개월 Rolling IC

기간별 IC 변화를 rolling 집계하여 신호의 안정성을 확인한다.

In [ ]:
# ============================================================
# 7. Horizon별 12개월 Rolling IC
# ============================================================

for signal in ["long_ic", "short_ic"]:

    pivot = (
        ic_df
        .pivot(
            index="date",
            columns="horizon",
            values=signal
        )
        .sort_index()
    )

    rolling = (
        pivot
        .rolling(12, min_periods=6)
        .mean()
    )

    rolling.columns = [
        "1M",
        "2M",
        "3M"
    ]

    plt.figure(figsize=(11, 5))

    for col in rolling.columns:

        plt.plot(
            rolling.index,
            rolling[col],
            linewidth=2,
            label=col
        )

    plt.axhline(
        0,
        linewidth=1
    )

    name = (
        "Long Score"
        if signal == "long_ic"
        else "Short Score"
    )

    plt.title(
        f"{name}: 12M Rolling Forward IC"
    )

    plt.ylabel("Rolling Spearman IC")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Signal Persistence

평균 IC를 기간별로 비교하여 신호 지속성의 근거를 검토한다.

In [ ]:
# ============================================================
# 8. Signal Persistence 간단 요약
# ============================================================

mean_ic = (
    ic_df
    .groupby("horizon")[
        ["long_ic", "short_ic"]
    ]
    .mean()
)

print("===== 평균 Forward IC =====")

for h in [1, 2, 3]:

    print(
        f"{h}개월: "
        f"Long IC = {mean_ic.loc[h, 'long_ic']:.4f}, "
        f"Short IC = {mean_ic.loc[h, 'short_ic']:.4f}"
    )


print("\n===== 3개월 신호 유지율 =====")

long_persistence = (
    mean_ic.loc[3, "long_ic"]
    / mean_ic.loc[1, "long_ic"]
)

short_persistence = (
    mean_ic.loc[3, "short_ic"]
    / mean_ic.loc[1, "short_ic"]
)

print(
    f"Long: 3M IC / 1M IC = "
    f"{long_persistence:.2%}"
)

print(
    f"Short: 3M IC / 1M IC = "
    f"{short_persistence:.2%}"
)

## 8. 월별 개별 미래 수익률

형성 시점 이후 각 월의 개별 수익률을 추출하는 보조 함수다.

In [ ]:
def get_month_specific_return(date, month_number):
    """
    형성시점 date 기준으로

    month_number = 1:
        t -> t+1 수익률

    month_number = 2:
        t+1 -> t+2 수익률

    month_number = 3:
        t+2 -> t+3 수익률
    """

    period = date.to_period("M")

    start_period = period + (month_number - 1)
    end_period = period + month_number

    if start_period not in date_of or end_period not in date_of:
        return None

    start_date = date_of[start_period]
    end_date = date_of[end_period]

    ret = (
        monthly_total.loc[end_date]
        / monthly_total.loc[start_date]
        - 1
    )

    return ret

## 9. 스코어 분포와 포트폴리오 선정 시각화

동일 유니버스·선정 규칙으로 롱·숏 점수 분포 및 선택 종목을 확인한다.

In [ ]:
# [시각화] 최근 리밸런싱 시점의 롱·숏 스코어 분포
# 이 셀은 위 백테스트와 같은 유니버스·팩터·선정 순서·5일 체결률 기준을 사용한다.
# 위쪽 패널은 점수 원값의 분포, 아래쪽 패널은 점수를 백분위 순위로 바꾼 분포다.
# 세로 위치는 점의 겹침을 줄이기 위한 고정 난수이며 투자 의미가 없다.

# 백테스트에서 실제로 종목을 다시 고른 형성일(start)과 첫 보유월 말(end)을 찾는다.
rebalance_dates = [
    (date_of[month - 1], date_of[month])
    for i, month in enumerate(months)
    if i % 3 == 0
]
score_date, score_next_date = rebalance_dates[-1]

# pick()과 동일하게 시가총액 상위 30% 및 모든 팩터가 존재하는 종목만 남긴다.
latest_market_cap = monthly_cap.loc[score_date].dropna()
latest_universe = latest_market_cap[
    latest_market_cap.rank(pct=True) >= 0.70
].index

score_stocks = pd.DataFrame({
    "mom12": mom12.loc[score_date],
    "mom6": mom6.loc[score_date],
    "vol": vol.loc[score_date],
    "skew": skew.loc[score_date],
    "mean": mean.loc[score_date],
}).dropna()

score_stocks = score_stocks.loc[
    score_stocks.index.intersection(latest_universe)
]
score_stocks = score_stocks.loc[
    price.loc[score_date].reindex(score_stocks.index).gt(0)
    & total.loc[score_date].reindex(score_stocks.index).gt(0)
]
score_stocks = score_stocks.loc[
    monthly_total.loc[score_next_date].reindex(score_stocks.index).notna()
]

# 위 전략 코드와 완전히 같은 방식으로 SHARPE126과 롱·숏 스코어를 계산한다.
score_stocks["sharpe"] = (
    score_stocks["mean"]
    / score_stocks["vol"].replace(0, np.nan)
)
score_stocks["long_score"] = (
    rank(score_stocks["mom6"])
    * rank(score_stocks["sharpe"])
    * (1 - rank(score_stocks["skew"]))
)
score_stocks["short_score"] = (
    rank(score_stocks["mom6"])
    * (1 - rank(score_stocks["vol"]))
    * (1 - rank(score_stocks["skew"]))
)

# 숏 하위 100개를 먼저 고르고, 이를 제외한 롱 적격 종목에서 롱 상위 50개를 고른다.
latest_short_candidates = list(
    score_stocks["short_score"].nsmallest(short_n).index
)
latest_long_ok = (
    score_stocks["sharpe"].notna()
    & fall_ok.loc[score_date].reindex(score_stocks.index).fillna(False)
)
latest_long_candidates = list(
    score_stocks.loc[latest_long_ok, "long_score"]
    .drop(latest_short_candidates, errors="ignore")
    .nlargest(long_n)
    .index 
)
# 기준 전략과 동일한 5일 체결률 80% 조건을 적용해 실제 최종 편입 종목을 구한다.
latest_liquidity = (
    monthly_adv.loc[score_date]
    .reindex(score_stocks.index)
    .fillna(0)
    .clip(lower=0)
)
latest_short_fill = (
    latest_liquidity * 0.10 * 5
    / (100_000_000_000 / short_n)
).clip(upper=1)
latest_long_fill = (
    latest_liquidity * 0.10 * 5
    / (100_000_000_000 / long_n)
).clip(upper=1)

latest_longs = [
    name for name in latest_long_candidates
    if latest_long_fill[name] >= 0.80
]
latest_shorts = [
    name for name in latest_short_candidates
    if latest_short_fill[name] >= 0.80
]


def plot_latest_score_distribution(
    score_column,
    korean_name,
    candidates,
    selected,
    direction,
    color,
    seed,
):
    """점수 원값과 백분위 순위 분포를 한 그림의 위·아래 패널에 표시한다."""
    scores = score_stocks[score_column].dropna().sort_values()
    percentile_scores = scores.rank(pct=True, method="average")

    # y축은 밀집된 점을 펼쳐 보이기 위한 고정 난수다.
    rng = np.random.default_rng(seed)
    jitter = pd.Series(
        rng.uniform(-0.72, 0.72, len(scores)),
        index=scores.index,
    )

    candidate_index = scores.index.intersection(candidates)
    selected_index = scores.index.intersection(selected)
    cutoff = (
        scores.loc[candidate_index].min()
        if direction == "high"
        else scores.loc[candidate_index].max()
    )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(14, 7.2),
        gridspec_kw={"height_ratios": [3.0, 1.05]},
    )
    raw_ax, rank_ax = axes

    # 전체 유니버스 → 점수 후보 → 유동성 통과 실제 편입 순으로 겹쳐 그린다.
    raw_ax.scatter(
        scores,
        jitter.loc[scores.index],
        s=25,
        color="#B8C1CC",
        alpha=0.42,
        linewidths=0,
        label=f"전체 유니버스 ({len(scores)}종목)",
    )
    raw_ax.scatter(
        scores.loc[candidate_index],
        jitter.loc[candidate_index],
        s=35,
        color=color,
        alpha=0.48,
        linewidths=0,
        label=f"점수 후보 ({len(candidate_index)}종목)",
    )
    raw_ax.scatter(
        scores.loc[selected_index],
        jitter.loc[selected_index],
        s=58,
        color=color,
        alpha=0.98,
        edgecolors="white",
        linewidths=0.6,
        label=f"5일 체결률 통과 ({len(selected_index)}종목)",
        zorder=4,
    )

    # 후보 선정 방향을 음영과 점선으로 표시한다.
    if direction == "high":
        raw_ax.axvspan(cutoff, 1.0, color=color, alpha=0.08)
    else:
        raw_ax.axvspan(0.0, cutoff, color=color, alpha=0.08)
    raw_ax.axvline(
        cutoff,
        color=color,
        linestyle="--",
        linewidth=1.7,
        alpha=0.9,
        label=f"후보 경계 {cutoff:.4f}",
    )

    raw_ax.axhline(0, color="#8793A1", linewidth=0.9, alpha=0.7)
    raw_ax.set_xlim(-0.015, 1.015)
    raw_ax.set_ylim(-0.95, 0.95)
    raw_ax.set_yticks([])
    raw_ax.set_xlabel(f"{korean_name} 원점수 (0~1)")
    raw_ax.grid(axis="x", alpha=0.24)
    raw_ax.legend(loc="upper right", frameon=False, ncol=2)

    low_01 = (scores <= 0.10).mean()
    low_02 = (scores <= 0.20).mean()
    raw_ax.set_title(
        f"전체 {len(scores)}종목  |  0~0.1: {low_01:.1%}  |  "
        f"0~0.2: {low_02:.1%}  |  후보 {len(candidate_index)}  |  "
        f"최종 편입 {len(selected_index)}",
        loc="left",
        fontsize=12,
        pad=12,
    )

    # 같은 점수를 백분위 순위로 변환하면 거의 균등한 간격으로 펼쳐지는지 확인할 수 있다.
    rank_ax.scatter(
        percentile_scores,
        jitter.loc[percentile_scores.index] * 0.16,
        s=18,
        color="#AAB4C0",
        alpha=0.48,
        linewidths=0,
    )
    rank_ax.scatter(
        percentile_scores.loc[selected_index],
        jitter.loc[selected_index] * 0.16,
        s=42,
        color=color,
        alpha=0.98,
        edgecolors="white",
        linewidths=0.5,
        zorder=4,
    )
    rank_ax.axhline(0, color="#8793A1", linewidth=0.9, alpha=0.7)
    rank_ax.set_xlim(-0.015, 1.015)
    rank_ax.set_ylim(-0.24, 0.24)
    rank_ax.set_yticks([])
    rank_ax.set_xlabel(f"{korean_name}의 횡단면 백분위 순위 (0~1)")
    rank_ax.set_title(
        "참고: 백분위 순위 변환 후 분포 - 진한 점은 실제 최종 편입 종목",
        loc="left",
        fontsize=11,
        pad=10,
    )
    rank_ax.grid(axis="x", alpha=0.24)

    fig.suptitle(
        f"{korean_name} 분포  |  최근 리밸런싱 형성일 {score_date:%Y-%m-%d}",
        fontsize=22,
        fontweight="bold",
        x=0.08,
        ha="left",
        y=1.02,
    )
    fig.text(
        0.08,
        0.005,
        "점 하나 = 종목 하나 · 세로 위치는 점 겹침을 줄이기 위한 고정 난수이며 별도 의미가 없습니다.",
        fontsize=10,
        color="#667281",
    )
    plt.tight_layout(rect=[0.03, 0.04, 0.99, 0.96])
    plt.show()


# 한글 제목이 깨지지 않도록 Windows 기본 한글 글꼴을 사용한다.
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
# 롱은 높은 점수가 후보 방향, 숏은 낮은 점수가 후보 방향이다.
plot_latest_score_distribution(
    score_column="long_score",
    korean_name="롱 스코어",
    candidates=latest_long_candidates,
    selected=latest_longs,
    direction="high",
    color="#2878B5",
    seed=930,
)

plot_latest_score_distribution(
    score_column="short_score",
    korean_name="숏 스코어",
    candidates=latest_short_candidates,
    selected=latest_shorts,
    direction="low",
    color="#D7653B",
    seed=931,
)

# 수치로도 확인할 수 있도록 최근 시점의 요약을 함께 표시한다.
score_distribution_summary = pd.DataFrame({
    "형성일": [score_date.date(), score_date.date()],
    "전체 종목 수": [
        score_stocks["long_score"].notna().sum(),
        score_stocks["short_score"].notna().sum(),
    ],
    "점수 후보 수": [len(latest_long_candidates), len(latest_short_candidates)],
    "최종 편입 수": [len(latest_longs), len(latest_shorts)],
    "최솟값": [score_stocks["long_score"].min(), score_stocks["short_score"].min()],
    "중앙값": [score_stocks["long_score"].median(), score_stocks["short_score"].median()],
    "최댓값": [score_stocks["long_score"].max(), score_stocks["short_score"].max()],
}, index=["롱 스코어", "숏 스코어"])

display(score_distribution_summary.round(4))
